Анализ результатов A/B-теста новой формы оплаты ЖКХ

Цель работы: понять, как новая форма с автозаполнением реквизитов повлияла на успешность оплаты ЖКХ, и решить, стоит ли раскатывать ее на всех пользователей приложения.

Воронка платежа состоит из четырех последовательных шагов:
1. Открыл форму оплаты
2. Ввел реквизиты
3. Подтвердил платеж
4. Успешная оплата

Главная метрика теста - конверсия из открытия формы в успешную оплату на уровне уникального пользователя. Сплитовали пользователей 50 на 50, тест шел 30 дней в апреле 2024 года.

План анализа:
1. Загрузка данных и первичный осмотр таблиц
2. Поиск аномалий, ошибок логирования и очистка датасета
3. Проверка качества сплитования (SRM-тест)
4. Расчет воронки по шагам для попыток и уникальных клиентов
5. Статистическая проверка гипотезы через Z-тест
6. Анализ эффекта по срезам (платформа, город, возраст)
7. Анализ времени прохождения шагов
8. Итоговые выводы и рекомендации для команды

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.proportion import proportions_ztest

Этап 1. Загрузка данных и первичный осмотр

Загружаем файл с данными. В нем две таблицы:
Лист Пользователи - идентификатор клиента, группа теста, возраст, город и операционная система.
Лист Платежи - попытки оплаты с метками времени для каждого шага воронки.

In [2]:
file_path = 'Данные для кейса по ПА.xlsx'

df_users_raw = pd.read_excel(file_path, sheet_name='Пользователи')
df_payments_raw = pd.read_excel(file_path, sheet_name='Платежи')

print("Пользователи:", df_users_raw.shape)
print(df_users_raw.head())

print()
print("Платежи:", df_payments_raw.shape)
print(df_payments_raw.head())

Пользователи: (1504, 5)
   user_id group  age    city device_type
0     1001     A   31  Москва         ios
1     1002     A   31  Москва         ios
2     1003     A   25  Москва     android
3     1004     A   20  Москва         ios
4     1005     A   34  Москва         ios

Платежи: (2661, 7)
   payment_id  user_id group        step1_opened       step2_entered  \
0        5001   1001.0     A 2024-04-16 06:06:00 2024-04-16 06:10:00   
1        5002   1002.0     A 2024-04-12 07:38:00 2024-04-12 07:41:00   
2        5003   1003.0     A 2024-04-26 20:03:00 2024-04-26 20:07:00   
3        5004   1004.0     A 2024-04-09 16:02:00                 NaT   
4        5005   1004.0     A 2024-04-04 03:46:00 2024-04-04 03:54:00   

      step3_confirmed       step4_success  
0                 NaT                 NaT  
1                 NaT                 NaT  
2                 NaT                 NaT  
3                 NaT                 NaT  
4 2024-04-04 03:57:00 2024-04-04 04:02:00  


Этап 2. Проверка качества данных и очистка

Смотрим, есть ли пропуски, битые значения возраста и конфликты в данных.

В таблице пользователей нашлись 4 проблемные строки:
- пользователь 9001 без группы теста
- пользователь 9002 с возрастом -5 лет
- пользователь 9003 с возрастом 150 лет
- пользователь 9004 без указания города

В таблице платежей тоже есть 3 проблемы:
- строка 99003 без user_id
- платеж 7658 от невалидного пользователя 9004
- платеж 99002, где у пользователя 1010 в пользователях группа A, а в платеже указана группа B

Исключаем эти некорректные записи, чтобы они не искажали результаты теста.

In [3]:
print("Пропуски в пользователях:")
print(df_users_raw.isnull().sum())

print()
print("Аномалии в возрасте:")
print(df_users_raw[(df_users_raw['age'] < 14) | (df_users_raw['age'] > 100)])

bad_users = df_users_raw[
    df_users_raw['group'].isnull() | 
    df_users_raw['city'].isnull() | 
    (df_users_raw['age'] < 14) | 
    (df_users_raw['age'] > 100)
]
print()
print("Пользователи на удаление:", len(bad_users))
print(bad_users)

print()
print("Пропуски в платежах:")
print(df_payments_raw.isnull().sum())

merged = df_payments_raw.merge(df_users_raw[['user_id', 'group']], on='user_id', suffixes=('_pay', '_user'))
mismatches = merged[merged['group_pay'] != merged['group_user']]
print()
print("Платежи с несовпадением группы:", len(mismatches))
print(mismatches[['payment_id', 'user_id', 'group_pay', 'group_user']])

Пропуски в пользователях:
user_id        0
group          1
age            0
city           1
device_type    0
dtype: int64

Аномалии в возрасте:
      user_id group  age    city device_type
1501     9002     A   -5     СПб     android
1502     9003     B  150  Регион         ios

Пользователи на удаление: 4
      user_id group  age    city device_type
1500     9001   NaN   25  Москва         ios
1501     9002     A   -5     СПб     android
1502     9003     B  150  Регион         ios
1503     9004     A   30     NaN     android

Пропуски в платежах:
payment_id            0
user_id               1
group                 0
step1_opened          0
step2_entered       405
step3_confirmed     942
step4_success      1139
dtype: int64

Платежи с несовпадением группы: 1
      payment_id  user_id group_pay group_user
2659       99002   1010.0         B          A


In [4]:
invalid_ids = set(bad_users['user_id'])
df_users = df_users_raw[~df_users_raw['user_id'].isin(invalid_ids)].copy()
df_users = df_users.dropna(subset=['group', 'city'])
df_users = df_users[(df_users['age'] >= 14) & (df_users['age'] <= 100)]

df_payments = df_payments_raw.dropna(subset=['user_id']).copy()
df_payments = df_payments[df_payments['user_id'].isin(df_users['user_id'])]
df_payments = df_payments.merge(df_users[['user_id', 'group']], on='user_id', suffixes=('_pay', ''))
df_payments = df_payments[df_payments['group_pay'] == df_payments['group']].drop(columns=['group_pay'])

Пользователи после очистки: 1500 (A: 750, B: 750)
Платежи после очистки: 2658 (A: 1307, B: 1351)


Этап 3. Проверка качества сплитования (SRM)

Проверяем, ровно ли система поделила трафик 50 на 50 и нет ли случайных перекосов по устройствам, городам или возрасту.
По размеру групп получаем ровно по 750 человек в каждой группе (p-value = 1.0).
По типам устройств p = 0.063, по городам p = 0.440, по возрасту p = 0.825. Значимых различий нет, группы сопоставимы и сплит качественный.

In [5]:
print("Сплит 50/50 p-val:", stats.chisquare(df_users['group'].value_counts()).pvalue)
print("Устройства p-val:", round(stats.chi2_contingency(pd.crosstab(df_users['device_type'], df_users['group']))[1], 4))
print("Города p-val:", round(stats.chi2_contingency(pd.crosstab(df_users['city'], df_users['group']))[1], 4))
a_age, b_age = df_users[df_users['group'] == 'A']['age'], df_users[df_users['group'] == 'B']['age']
print("Возраст p-val:", round(stats.ttest_ind(a_age, b_age).pvalue, 4))

Сплит 50/50 p-val: 1.0
Устройства p-val: 0.0629
Города p-val: 0.4404
Возраст p-val: 0.8245


Этап 4. Построение продуктовой воронки

Считаем воронку на двух уровнях:
1. На уровне попыток (всего сессий оплаты)
2. На уровне уникальных пользователей (дошел ли клиент до успешной оплаты хотя бы один раз)

Так как сплитовали именно пользователей, ключевая метрика для бизнес-решения - это конверсия на уровне пользователей.

По шагам видно, где именно сработала новая форма:
- Конверсия из шага 1 в шаг 2 (ввод реквизитов) выросла с 81.1% до 88.4% (+7.3 п.п.). Людям больше не нужно вручную перепечатывать длинные лицевые счета.
- Конверсия из шага 2 в шаг 3 (подтверждение) выросла с 71.4% до 80.4% (+9.0 п.п.). Автоматическая подтяжка суммы и поставщика убирает неуверенность и страх ошибки.
- Суммарная конверсия пользователей выросла с 65.6% до 80.4% (+14.8 п.п.).

In [6]:
# Воронка на уровне попыток
for grp in ['A', 'B']:
    sub = df_payments[df_payments['group'] == grp]
    s1 = sub['step1_opened'].count()
    s2 = sub['step2_entered'].count()
    s3 = sub['step3_confirmed'].count()
    s4 = sub['step4_success'].count()
    print(f"Попытки {grp}: {s1} -> {s2} ({s2/s1:.2%}) -> {s3} ({s3/s2:.2%}) -> {s4} ({s4/s3:.2%}) | Итог: {s4/s1:.2%}")

# Воронка на уровне пользователей
cols = ['step1_opened', 'step2_entered', 'step3_confirmed', 'step4_success']
user_steps = df_payments.groupby('user_id')[cols].count().gt(0).astype(int).reset_index()
df_users_funnel = df_users.merge(user_steps, on='user_id', how='left').fillna(0)

print()
print("Пользователи:")
for grp in ['A', 'B']:
    sub = df_users_funnel[df_users_funnel['group'] == grp]
    tot = len(sub)
    u1 = sub['step1_opened'].sum()
    u2 = sub['step2_entered'].sum()
    u3 = sub['step3_confirmed'].sum()
    u4 = sub['step4_success'].sum()
    print(f"Группа {grp}: всего {tot}, открыли {u1}, реквизиты {u2}, подтвердили {u3}, оплатили {u4} | CR: {u4/tot:.2%}")

Попытки A: 1307 -> 1060 (81.10%) -> 757 (71.42%) -> 650 (85.87%) | Итог: 49.73%
Попытки B: 1351 -> 1194 (88.38%) -> 960 (80.40%) -> 870 (90.62%) | Итог: 64.40%

Пользователи:
Группа A: всего 750, открыли 750, реквизиты 670, подтвердили 547, оплатили 492 | CR: 65.60%
Группа B: всего 750, открыли 750, реквизиты 707, подтвердили 644, оплатили 603 | CR: 80.40%


Этап 5. Статистическая проверка продуктовой гипотезы

Формулировка гипотез:
Нулевая гипотеза H0: доля успешных пользователей в группе B такая же, как в группе A.
Альтернативная гипотеза H1: доля успешных пользователей в группе B отличается от группы A.

Для сравнения двух независимых долей используем двухвыборочный Z-критерий. Выборки достаточно большие (по 750 человек), условия нормальной аппроксимации выполняются.
Также строим 95% доверительный интервал для разницы долей.

In [7]:
s_a = df_users_funnel[df_users_funnel['group'] == 'A']['step4_success'].sum()
s_b = df_users_funnel[df_users_funnel['group'] == 'B']['step4_success'].sum()
n_a, n_b = 750, 750

p_a = s_a / n_a
p_b = s_b / n_b
diff = p_b - p_a
lift = diff / p_a

z_stat, p_val = proportions_ztest([s_b, s_a], [n_b, n_a])

se = np.sqrt(p_a * (1 - p_a) / n_a + p_b * (1 - p_b) / n_b)
ci = (diff - 1.96 * se, diff + 1.96 * se)

print(f"CR A: {p_a:.2%}, CR B: {p_b:.2%}")
print(f"Прирост: +{diff*100:.2f} п.п. (lift: +{lift*100:.2f}%)")
print(f"z-stat: {z_stat:.3f}, p-value: {p_val:.4e}")
print(f"95% ДИ: [{ci[0]*100:.2f}%, {ci[1]*100:.2f}%]")

CR A: 65.60%, CR B: 80.40%
Прирост: +14.80 п.п. (lift: +22.56%)
z-stat: 6.456, p-value: 1.0782e-10
95% ДИ: [10.37%, 19.23%]


Значение p-value существенно меньше 0.05 (порядка 10 в минус 10 степени). Доверительный интервал строго выше нуля: от +10.37% до +19.23%.
Нулевая гипотеза уверенно отвергается, эффект статистически достоверен.

Этап 6. Сегментный анализ

Проверяем устойчивость эффекта по срезам пользователей: платформам (iOS / Android), городам и возрастным когортам.
Важно убедиться, что ни в одном сегменте нет просадки.

In [8]:
df_users_funnel['age_group'] = pd.cut(
    df_users_funnel['age'], 
    bins=[0, 24, 34, 44, 120], 
    labels=['< 25', '25-34', '35-44', '45+']
)

for col in ['device_type', 'city', 'age_group']:
    print(f"\nСрез: {col}")
    for val in sorted(df_users_funnel[col].unique()):
        sub_a = df_users_funnel[(df_users_funnel[col] == val) & (df_users_funnel['group'] == 'A')]
        sub_b = df_users_funnel[(df_users_funnel[col] == val) & (df_users_funnel['group'] == 'B')]
        n_a, s_a = len(sub_a), sub_a['step4_success'].sum()
        n_b, s_b = len(sub_b), sub_b['step4_success'].sum()
        cr_a = s_a / n_a if n_a else 0
        cr_b = s_b / n_b if n_b else 0
        z, p = proportions_ztest([s_b, s_a], [n_b, n_a]) if (n_a and n_b) else (0, 1)
        print(f"{val}: A={cr_a:.1%} ({s_a}/{n_a}), B={cr_b:.1%} ({s_b}/{n_b}), diff={cr_b-cr_a:+.2%}, p={p:.4f}")


Срез: device_type
android: A=64.3% (261/406), B=79.9% (295/369), diff=+15.66%, p=0.0000
ios: A=67.2% (231/344), B=80.8% (308/381), diff=+13.69%, p=0.0000

Срез: city
Москва: A=62.5% (180/288), B=81.0% (248/306), diff=+18.55%, p=0.0000
Регион: A=66.8% (179/268), B=76.3% (187/245), diff=+9.54%, p=0.0170
СПб: A=68.6% (133/194), B=84.4% (168/199), diff=+15.87%, p=0.0002

Срез: age_group
25-34: A=64.9% (211/325), B=81.0% (239/295), diff=+16.09%, p=0.0000
35-44: A=62.9% (151/240), B=78.9% (210/266), diff=+16.03%, p=0.0001
45+: A=68.9% (42/61), B=75.6% (34/45), diff=+6.70%, p=0.4489
< 25: A=71.0% (88/124), B=83.3% (120/144), diff=+12.37%, p=0.0155


Выводы по сегментам:
Эффект стабильно положительный во всех срезах без исключений.
- По операционным системам: Android (+15.7 п.п.) и iOS (+13.7 п.п.).
- По географии наибольший прирост в Москве (+18.5 п.п.) и Петербурге (+15.9 п.п.), в регионах прирост тоже значимый (+9.5 п.п.).
- По возрасту максимальный эффект у активной аудитории 25-34 (+16.1 п.п.) и 35-44 (+16.0 п.п.). В когорте 45+ лет выборка небольшая (106 человек), прирост составил +6.7 п.п., но p-value не достиг значимости из-за размера выборки.

Этап 7. Анализ времени прохождения воронки

Проверяем, изменилось ли время заполнения формы и полного цикла оплаты у тех, кто успешно совершил платеж.
Смотрим время от открытия формы до ввода реквизитов и от открытия формы до успешной оплаты.

In [9]:
df_success = df_payments[df_payments['step4_success'].notnull()].copy()
df_success['t_1_2'] = (df_success['step2_entered'] - df_success['step1_opened']).dt.total_seconds() / 60
df_success['t_1_4'] = (df_success['step4_success'] - df_success['step1_opened']).dt.total_seconds() / 60

for grp in ['A', 'B']:
    sub = df_success[df_success['group'] == grp]
    print(f"Группа {grp}:")
    print(f"  Ввод реквизитов (мин): медиана {sub['t_1_2'].median():.1f}, среднее {sub['t_1_2'].mean():.2f}")
    print(f"  Весь платеж (мин): медиана {sub['t_1_4'].median():.1f}, среднее {sub['t_1_4'].mean():.2f}")

u_stat, p_time = stats.mannwhitneyu(
    df_success[df_success['group'] == 'A']['t_1_4'], 
    df_success[df_success['group'] == 'B']['t_1_4']
)
print(f"\nMann-Whitney p-value (общее время): {p_time:.4f}")

Группа A:
  Ввод реквизитов (мин): медиана 1.0, среднее 2.37
  Весь платеж (мин): медиана 12.0, среднее 12.42
Группа B:
  Ввод реквизитов (мин): медиана 1.0, среднее 2.43
  Весь платеж (мин): медиана 12.0, среднее 12.35

Mann-Whitney p-value (общее время): 0.4085


Медианы времени в обеих группах абсолютно одинаковы: 1 минута на ввод реквизитов и 12 минут на весь платеж. Тест Манна-Уитни подтверждает отсутствие значимых различий (p = 0.409).
Это важный продуктовый вывод: автозаполнение экономит не секунды, а снимает когнитивную нагрузку и страх ошибиться в длинных реквизитах, благодаря чему клиенты доводят оплату до конца.

Этап 8. Итоговые выводы и рекомендации для команды

Главные результаты:
1. Новая форма с автозаполнением показала сильный статистически значимый рост конверсии: с 65.6% до 80.4% (+14.8 п.п., относительный прирост +22.6%, p < 0.0001).
2. Форма убрала отвалы именно на шагах поиска квитанции и ввода реквизитов (+7.3 п.п. и +9.0 п.п.).
3. Положительный эффект устойчив во всех сегментах и на обеих мобильных платформах.

Бизнес-решение:
Запускать новую форму оплаты с автозаполнением на 100% пользователей.

План раскатки и контроль рисков:
1. Раскатывать поэтапно (25% -> 50% -> 100% с интервалом в 2-3 дня), чтобы отследить нагрузку на внешние шлюзы поставщиков ЖКХ в дни пиковых оплат.
2. Мониторить обращения в техподдержку по теме некорректных начислений или неверно подтянутых реквизитов.
3. Для пользователей старшего возраста (45+) сохранить на первом экране явную кнопку ручной корректировки данных, чтобы консервативные клиенты чувствовали контроль над операцией.